# Multi-seed training

Trains every condition with several random seeds and saves the class
probabilities (validation set only at the `"val"` stage; validation and test
sets at the `"final"` stage). Late fusion and all statistics (CIs, paired
comparisons, B2/C1 analysis) are done afterwards in `02_analysis`.

Conditions are named `{head}_{features}`:

- **head**
  - `hf`: Hugging Face's own classification head (dense → GELU → LayerNorm →
    dropout → linear), with the CEFR vector joined just before the final
    linear layer. `hf_only` is identical to `ModernBertForSequenceClassification`.
    These are the conditions reported in the paper.
  - `tanh`: the custom head used in the first version of the study
    (dense → tanh → linear on [CLS; CEFR vector]). Trained as a check that the
    head does not affect the results; not reported in the paper.
- **features**: `only`, `AW_raw`, `AW_std`, `LazyA1_raw`, `LazyA1_std`
  (raw = proportions as originally computed; std = standardised using the
  training set)
- `vocab_AW` / `vocab_LazyA1`: MLP trained on the proportions only

Set `STAGE` below: `"val"` while developing, `"final"` once at the end.
Runs are cached: if a result file already exists, that run is skipped, so the
notebook can be stopped and restarted.

In [3]:
import gc
import json
from pathlib import Path

import numpy as np
import torch
from torch import nn
from datasets import load_dataset
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from transformers import (
    AutoModel,
    AutoTokenizer,
    DataCollatorWithPadding,
    ModernBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)
from transformers.modeling_outputs import SequenceClassifierOutput

W1021 08:18:21.869000 28800 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1021 08:18:21.951000 28800 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [4]:
# ---- Settings -------------------------------------------------------------
DATA_DIR = Path(r"C:\Users\coope\Documents\Learner_Writing_Classification\cefr_lexical_sophistication\cefr-lexical-sophistication-main\wai_data\cefr_vocab_data")
# STAGE = "val"   : development. Only the validation set is predicted; the test
#                   set is never passed through a model. Use this to debug and to
#                   settle every choice (seeds, conditions, analysis).
# STAGE = "final" : run once at the end. Retrains everything from scratch and
#                   predicts validation and test. Results go to a separate folder,
#                   so the development results are kept.
STAGE = "val"
PREDICT_SPLITS = ["val"] if STAGE == "val" else ["val", "test"]

OUT_DIR = Path("robustness_outputs")
PROB_DIR = OUT_DIR / f"probs_{STAGE}"
PROB_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "answerdotai/ModernBERT-base"
SEEDS = [42, 1, 2, 3, 4, 5, 6, 7, 8, 9]   # 10 seeds; 5 is the minimum worth reporting
LEARNING_RATE = 1e-05                      # same as the paper
EPOCHS = 3
BATCH_SIZE = 8                             # HF default, as in the paper

HEADS = ["hf", "tanh"]        # remove "tanh" to halve the run time (main analysis uses "hf")
FEATURES = ["only", "AW_raw", "AW_std", "LazyA1_raw", "LazyA1_std"]
BERT_CONDITIONS = [f"{h}_{f}" for h in HEADS for f in FEATURES]
VOCAB_MODES = ["AW", "LazyA1"]

LABEL_MAP = {"A2": 0, "B1": 1, "B2": 2, "C1": 3}
CEFR_KEYS = ["cefr_A1", "cefr_A2", "cefr_B1", "cefr_B2", "cefr_C1", "cefr_C2"]

In [3]:
# ---- Load data ------------------------------------------------------------
# Texts and labels are identical in the AW and LazyA1 files; only the
# vocabulary vectors differ. Both are loaded and checked for alignment.
def load_split(split, mode):
    ds = load_dataset("json", data_files=str(DATA_DIR / f"{split}_cefr_{mode}.jsonl"), split="train")
    ds = ds.filter(lambda x: x["label"] in LABEL_MAP)
    return ds

data = {}
for split in ["train"] + PREDICT_SPLITS:          # test file is not even loaded at the "val" stage
    aw = load_split(split, "AW")
    lazy = load_split(split, "LazyA1")
    assert list(aw["id"]) == list(lazy["id"]), f"{split}: AW and LazyA1 files are not in the same order"
    data[split] = {
        "id": list(aw["id"]),
        "text": list(aw["text"]),
        "labels": np.array([LABEL_MAP[l] for l in aw["label"]]),
        "vec_AW": np.array([[x[k] for k in CEFR_KEYS] for x in aw], dtype=np.float32),
        "vec_LazyA1": np.array([[x[k] for k in CEFR_KEYS] for x in lazy], dtype=np.float32),
    }
    print(split, len(data[split]["labels"]), np.bincount(data[split]["labels"]))

# Standardised versions: scaler fitted on the training set only
for mode in VOCAB_MODES:
    scaler = StandardScaler().fit(data["train"][f"vec_{mode}"])
    for split in data:
        data[split][f"vec_{mode}_std"] = scaler.transform(data[split][f"vec_{mode}"]).astype(np.float32)

# Save ids and gold labels once, for the analysis notebook
np.savez(OUT_DIR / f"gold_{STAGE}.npz",
         **{f"{sp}_labels": data[sp]["labels"] for sp in PREDICT_SPLITS},
         **{f"{sp}_ids": np.array(data[sp]["id"], dtype=str) for sp in PREDICT_SPLITS})

train 4195 [ 876 1717 1225  377]
val 401 [ 86 157 111  47]


In [4]:
# ---- Tokenise once --------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
enc = {split: tokenizer(data[split]["text"], truncation=True, max_length=8192) for split in data}

class FeatureDataset(torch.utils.data.Dataset):
    """input_ids / attention_mask / labels, plus an optional CEFR vector."""
    def __init__(self, split, vec_key=None):
        self.enc = enc[split]
        self.labels = data[split]["labels"]
        self.vecs = data[split][vec_key] if vec_key else None

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {"input_ids": self.enc["input_ids"][i],
                "attention_mask": self.enc["attention_mask"][i],
                "labels": int(self.labels[i])}
        if self.vecs is not None:
            item["cefr_vec"] = self.vecs[i].tolist()
        return item

class CefrCollator(DataCollatorWithPadding):
    def __call__(self, features):
        vecs = [f.pop("cefr_vec") for f in features] if "cefr_vec" in features[0] else None
        batch = super().__call__(features)
        if vecs is not None:
            batch["cefr_vec"] = torch.tensor(vecs, dtype=torch.float)
        return batch

collator = CefrCollator(tokenizer=tokenizer)

In [5]:
# ---- Model: Hugging Face head ----------------------------------------------
# Wraps a genuine ModernBertForSequenceClassification. The encoder, pooling,
# head (dense -> GELU -> LayerNorm) and dropout are HF's own modules, loaded and
# initialised by HF. The only change is that the final linear layer takes
# [head output ; CEFR vector], initialised the way HF initialises it.
# With n_extra = 0 the stock model is called directly (identical to the paper's
# baseline). Because the head ends in a LayerNorm, its 768 outputs are on a
# unit scale, so the standardised CEFR vector (std) sits on the same scale.
# (A subclass is avoided on purpose: transformers v5 skips ModernBERT's own
# head initialisation for user-defined subclasses.)
class ModernBertHFWithVocab(nn.Module):
    def __init__(self, n_extra, num_labels=4):
        super().__init__()
        self.base = ModernBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
        self.n_extra = n_extra
        if n_extra:
            cfg = self.base.config
            std, cut = cfg.hidden_size ** -0.5, (cfg.initializer_cutoff_factor or 3)
            self.base.classifier = nn.Linear(cfg.hidden_size + n_extra, num_labels)
            nn.init.trunc_normal_(self.base.classifier.weight, std=std, a=-cut * std, b=cut * std)
            nn.init.zeros_(self.base.classifier.bias)

    def forward(self, input_ids=None, attention_mask=None, cefr_vec=None, labels=None):
        if not self.n_extra:
            return self.base(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        b = self.base
        hidden = b.model(input_ids=input_ids, attention_mask=attention_mask)[0]
        if b.config.classifier_pooling == "cls":
            pooled = hidden[:, 0]
        else:
            pooled = (hidden * attention_mask.unsqueeze(-1)).sum(1) / attention_mask.sum(1, keepdim=True)
        pooled = b.drop(b.head(pooled))
        pooled = torch.cat([pooled, cefr_vec.to(pooled.dtype)], dim=-1)
        logits = b.classifier(pooled)
        loss = nn.functional.cross_entropy(logits, labels) if labels is not None else None
        return SequenceClassifierOutput(loss=loss, logits=logits)

# ---- Model: the paper's tanh head ------------------------------------------
# Same head as the paper's CustomSequenceClassification (dense -> tanh -> out
# on [CLS ; cefr_vec]). tanh_only has no extra dims.
class ClassificationHead(nn.Module):
    def __init__(self, in_dims, num_labels, dropout=0.0):
        super().__init__()
        self.dense = nn.Linear(in_dims, in_dims)
        self.dropout = nn.Dropout(dropout)
        self.out_proj = nn.Linear(in_dims, num_labels)

    def forward(self, x):
        x = self.dropout(x)
        x = torch.tanh(self.dense(x))
        x = self.dropout(x)
        return self.out_proj(x)

class ModernBertWithVocab(nn.Module):
    def __init__(self, n_extra, num_labels=4):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        cfg = self.encoder.config
        self.n_extra = n_extra
        self.head = ClassificationHead(cfg.hidden_size + n_extra, num_labels, cfg.classifier_dropout)
        for m in self.head.modules():                        # same init as HF post_init
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=cfg.initializer_range)
                nn.init.zeros_(m.bias)

    def forward(self, input_ids=None, attention_mask=None, cefr_vec=None, labels=None):
        hidden = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        x = hidden[:, 0, :]
        if self.n_extra:
            x = torch.cat([x, cefr_vec.to(x.dtype)], dim=-1)
        logits = self.head(x)
        loss = nn.functional.cross_entropy(logits, labels) if labels is not None else None
        return SequenceClassifierOutput(loss=loss, logits=logits)

def build_model(condition):
    head, feats = condition.split("_", 1)                # e.g. "hf", "AW_std"
    if feats == "only":
        n_extra, vec_key = 0, None
    else:
        mode, scaling = feats.split("_")
        n_extra, vec_key = 6, f"vec_{mode}" + ("_std" if scaling == "std" else "")
    if head == "hf":
        model = ModernBertHFWithVocab(n_extra=n_extra)
    else:
        model = ModernBertWithVocab(n_extra=n_extra)
    return model, vec_key

In [6]:
# ---- Training loop --------------------------------------------------------
def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def run_bert(condition, seed):
    out_file = PROB_DIR / f"{condition}_seed{seed}.npz"
    if out_file.exists():
        return
    set_seed(seed)                                       # controls head init + data order
    model, vec_key = build_model(condition)

    args = TrainingArguments(
        output_dir=str(OUT_DIR / "tmp"),
        save_strategy="no",
        report_to="none",
        seed=seed,
        data_seed=seed,
        learning_rate=LEARNING_RATE,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=16,
        remove_unused_columns=False,
        logging_steps=200,
    )
    trainer = Trainer(model=model, args=args, data_collator=collator,
                      train_dataset=FeatureDataset("train", vec_key))
    trainer.train()

    probs = {}
    for split in PREDICT_SPLITS:
        logits = trainer.predict(FeatureDataset(split, vec_key)).predictions
        logits = logits[0] if isinstance(logits, tuple) else logits
        probs[split] = softmax(logits)
    np.savez(out_file, **probs)

    acc = (probs["val"].argmax(1) == data["val"]["labels"]).mean()
    print(f"{condition} seed {seed}: validation accuracy {acc:.3f}")

    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()

def run_vocab_only(mode, seed):
    out_file = PROB_DIR / f"vocab_{mode}_seed{seed}.npz"
    if out_file.exists():
        return
    clf = make_pipeline(StandardScaler(),
                        MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=500, random_state=seed))
    clf.fit(data["train"][f"vec_{mode}"], data["train"]["labels"])
    np.savez(out_file, **{split: clf.predict_proba(data[split][f"vec_{mode}"]) for split in PREDICT_SPLITS})

In [7]:
# Written before training so 02_analysis.py works even if you stop early
json.dump({"stage": STAGE, "splits": PREDICT_SPLITS, "seeds": SEEDS, "heads": HEADS, "features": FEATURES,
           "bert_conditions": BERT_CONDITIONS, "vocab_modes": VOCAB_MODES,
           "lr": LEARNING_RATE, "epochs": EPOCHS, "batch_size": BATCH_SIZE},
          open(OUT_DIR / f"run_config_{STAGE}.json", "w"), indent=2)

In [8]:
# Vocabulary-only models (seconds)
for seed in SEEDS:
    for mode in VOCAB_MODES:
        run_vocab_only(mode, seed)

c:\Users\coope\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [9]:
# ModernBERT models (about 2 min per run on 5090 GPU; 10 conditions x 10 seeds =
# roughly 3.5 hours, or half that with HEADS = ["hf"]).
# Loops seed-first so that if you stop early, every condition has the same seeds.
for seed in SEEDS:
    for condition in BERT_CONDITIONS:
        run_bert(condition, seed)

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
200,0.952636
400,0.742395
600,0.662338
800,0.550355
1000,0.555112
1200,0.456728
1400,0.394894


hf_only seed 42: validation accuracy 0.658


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

...

tanh_LazyA1_raw seed 9: validation accuracy 0.628


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss
200,0.967087
400,0.837104
600,0.762857
800,0.667021
1000,0.658785
1200,0.576266
1400,0.580380


tanh_LazyA1_std seed 9: validation accuracy 0.636


In [5]:
print("Done:", sorted(p.name for p in PROB_DIR.glob("*.npz")))

Done: ['hf_AW_raw_seed1.npz', 'hf_AW_raw_seed2.npz', 'hf_AW_raw_seed3.npz', 'hf_AW_raw_seed4.npz', 'hf_AW_raw_seed42.npz', 'hf_AW_raw_seed5.npz', 'hf_AW_raw_seed6.npz', 'hf_AW_raw_seed7.npz', 'hf_AW_raw_seed8.npz', 'hf_AW_raw_seed9.npz', 'hf_AW_std_seed1.npz', 'hf_AW_std_seed2.npz', 'hf_AW_std_seed3.npz', 'hf_AW_std_seed4.npz', 'hf_AW_std_seed42.npz', 'hf_AW_std_seed5.npz', 'hf_AW_std_seed6.npz', 'hf_AW_std_seed7.npz', 'hf_AW_std_seed8.npz', 'hf_AW_std_seed9.npz', 'hf_LazyA1_raw_seed1.npz', 'hf_LazyA1_raw_seed2.npz', 'hf_LazyA1_raw_seed3.npz', 'hf_LazyA1_raw_seed4.npz', 'hf_LazyA1_raw_seed42.npz', 'hf_LazyA1_raw_seed5.npz', 'hf_LazyA1_raw_seed6.npz', 'hf_LazyA1_raw_seed7.npz', 'hf_LazyA1_raw_seed8.npz', 'hf_LazyA1_raw_seed9.npz', 'hf_LazyA1_std_seed1.npz', 'hf_LazyA1_std_seed2.npz', 'hf_LazyA1_std_seed3.npz', 'hf_LazyA1_std_seed4.npz', 'hf_LazyA1_std_seed42.npz', 'hf_LazyA1_std_seed5.npz', 'hf_LazyA1_std_seed6.npz', 'hf_LazyA1_std_seed7.npz', 'hf_LazyA1_std_seed8.npz', 'hf_LazyA1_std_s